# Experimento 4 — Regressão Logística com balanceamento utilizando todas as variáveis

Este quarto experimento combina as abordagens testadas nas etapas anteriores: a expansão do conjunto de atributos (*features*) com os parâmetros físico-químicos diretos (Experimento 3) e a aplicação da penalidade matemática na função de custo através do parâmetro `class_weight='balanced'` (Experimento 2).

O objetivo é verificar se a oferta de um contexto químico mais rico (incluindo DBO, OD, pH e Amônia) fornece à Regressão Logística a separabilidade linear necessária para identificar as classes minoritárias (`Crítica` e `Atenção`) sem sacrificar tão drasticamente a sua precisão (reduzindo os falsos alarmes gerados no Experimento 2).

**Variáveis utilizadas (Todas padronizadas ou codificadas):**
- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `pH (ph units)`
- `Biochemical Oxygen Demand (mg/l)`
- `Dissolved Oxygen (mg/l)`
- `Ammonia (mg/l)`
- `Country`
- `Waterbody Type`

**Variável alvo:**
- `conama_status` (4 classes)

## Preparação do ambiente

In [2]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [3]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/dataset_pisi3/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [4]:
# PREPARANDO O AMBIENTE PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [5]:
# DEFINIÇÃO DE X E Y (BASE COMPLETA DE VARIÁVEIS)
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "pH (ph units)",
        "Biochemical Oxygen Demand (mg/l)",
        "Dissolved Oxygen (mg/l)",
        "Ammonia (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [6]:
# TRAIN TEST SPLIT WITH STRATIFY
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 8)
Teste: (28280, 8)


In [7]:
# PRÉ-PROCESSAMENTO (APLICANDO SCALER NAS 6 VARIÁVEIS NUMÉRICAS)
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Ammonia (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [8]:
# CONSTRUÇÃO DO PIPELINE (COM CLASS_WEIGHT BALANCED REATIVADO)
model_balanced_all = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=SEED,
                max_iter=1000,
                multi_class='multinomial',
                class_weight='balanced' # <-- Ajuste de pesos ativado
            )
        )
    ]
)

In [9]:
model_balanced_all.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type']),
                                                 ('num', StandardScaler(),
                                                  ['Temperature (cel)',
                                                   'Orthophosphate (mg/l)',
                                                   'pH (ph units)',
                                                   'Biochemical Oxygen Demand '
                                                   '(mg/l)',
                                                   'Dissolved Oxygen (mg/l)',
                                                   'Ammonia (mg/l)'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    multi_class='multinomial',
                                    random_state=42))])

## Avaliação das Métricas de Treino

In [10]:
y_train_pred = model_balanced_all.predict(X_train)

In [11]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:\n", train_accuracy)
print("\nTrain Precision:\n", train_precision)
print("\nTrain Recall:\n", train_recall)
print("\nTrain F1:\n", train_f1)
print("\nTrain Confusion Matrix:\n", train_confusion_matrix)

Train Accuracy:
 0.7912729072923205

Train Precision:
 0.8175122215956815

Train Recall:
 0.7912729072923205

Train F1:
 0.8010931268213443

Train Confusion Matrix:
 [[ 3371  1689  2453    47]
 [ 1473 10899  2127  7179]
 [  326    86   692     2]
 [   32  7015  1182 74546]]


## Avaliação no Conjunto de Teste

In [12]:
y_pred_bal_all = model_balanced_all.predict(X_test)

print("Accuracy:\n", accuracy_score(y_test, y_pred_bal_all))

print("\nClassification Report:\n", classification_report(y_test, y_pred_bal_all, zero_division=0))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_bal_all))

Accuracy:
 0.7896746817538897

Classification Report:
               precision    recall  f1-score   support

     Atenção       0.65      0.44      0.53      1890
         Boa       0.55      0.49      0.52      5419
     Crítica       0.11      0.64      0.18       277
   Excelente       0.91      0.90      0.91     20694

    accuracy                           0.79     28280
   macro avg       0.56      0.62      0.53     28280
weighted avg       0.82      0.79      0.80     28280


Confusion Matrix:
 [[  834   427   622     7]
 [  358  2670   584  1807]
 [   78    21   178     0]
 [    5  1735   304 18650]]


# Resultados Obtidos e Considerações — Experimento 4 (RL Todas as Variáveis + Balanceamento)

## O Efeito Combinado (Química + Matemática)
A união das variáveis físico-químicas (Experimento 3) com o ajuste de pesos na função de custo (Experimento 2) resultou em uma acurácia global de **0.6974** (~69.7%) e um F1-Score ponderado de **0.71**.

Embora a acurácia seja menor do que a obtida no modelo sem balanceamento (~78%), essa queda é um compromisso esperado e desejado. O modelo abriu mão de classificar tudo como `Excelente` (o que inflava a nota) para focar na detecção real das anomalias hídricas.

## A Recuperação das Classes Críticas (O Grande Avanço)
A combinação de domínio ambiental e matemática penalizada trouxe avanços expressivos nas classes minoritárias:
* **Classe Crítica:** O *Recall* atingiu **0.77**. O modelo agora consegue identificar 77% de todas as amostras de água em estado crítico (um salto imenso comparado ao zero absoluto do Experimento 3). A *Precision* subiu para **0.15** (quase o dobro do Experimento 2), mostrando que a injeção de novas variáveis ajudou a reduzir a taxa de falsos alarmes.
* **Classe Atenção:** O *Recall* subiu para **0.53** e a precisão para **0.28**, demonstrando uma capacidade intermediária de rastrear níveis primários de degradação.

## Análise da Matriz de Confusão (Falha Segura)
A matriz de confusão deste experimento revela um comportamento crucial para um sistema de monitoramento ambiental: **a falha segura**.
* **Zero Falsos Negativos Graves:** Das 277 amostras reais da classe `Crítica` (terceira linha), o modelo identificou 213 perfeitamente e **nenhuma (0) amostra crítica foi classificada como Excelente**. Do ponto de vista de saúde pública e gestão hídrica, é preferível que o modelo erre classificando águas limpas como poluídas do que o inverso.
* **Falsos Positivos da Fronteira Linear:** O preço pago para zerar os erros graves foi empurrar a fronteira de decisão. Observa-se que 1.148 amostras realmente `Excelente` (última linha) acabaram sendo classificadas como `Crítica`. A Regressão Logística, por ser linear, teve que traçar uma linha "agressiva" no espaço de dados para englobar os casos ruins.

## Conclusão da Etapa
O Experimento 4 prova inequivocamente que a arquitetura correta para o dataset AquaSense exige tanto a riqueza de variáveis biológicas quanto técnicas rígidas de balanceamento. A Regressão Logística encontrou o seu melhor cenário possível aqui, atuando como um filtro conservador de qualidade da água.

A alta taxa residual de falsos positivos confirma que atingimos o limite estrutural do modelo linear. A transição para modelos não lineares (Experimentos 5 e 6, SMOTE e Tuning) ditará se conseguimos refinar a precisão sem sacrificar esse excelente ganho de *Recall*.

In [13]:
# Salvar os resultados para a nossa tabela comparativa geral
test_accuracy = accuracy_score(y_test, y_pred_bal_all)
test_f1 = f1_score(y_test, y_pred_bal_all, average="weighted", zero_division=0)

resultados = {
    "experimento": "exp_04_logistic_regression_todas_variaveis_balanceado",
    "algoritmo": "LogisticRegression",
    "balanceamento": "class_weight_balanced",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/EDA_AquaSense/resultados_experimentos.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas do Experimento 4 salvas com sucesso.")

Métricas do Experimento 4 salvas com sucesso.
